# 12b — View FITS Cutouts for dipole-rich diaObjects

## Purpose

Visualise the triplet **(Science / Template / Difference)** for every diaSource
of a selected `diaObjectId`, reading **FITS files** from
`12a_downloadSelectedCutoutsFits.ipynb`.

Each stamp has WCS axes (RA/Dec) and four overlays:

| Arrow | Colour | Meaning |
|-------|--------|---------|
| **N** | red    | North (+Dec) via WCS.world_to_pixel_values |
| **E** | blue   | East (+RA cosδ) via WCS.world_to_pixel_values |
| **Z** | yellow | Zenith direction (parallactic angle η) |
| **Dip** | cyan ↔ | Dipole axis = `r:dipoleAngle` (diff panel only) |

Layout per diaSource — **2 × 3 grid**:

| | col 0 | col 1 | col 2 |
|---|---|---|---|
| row 0 | Info panel | Science stamp | Template stamp |
| row 1 | Light curve | DIA Difference | Sci − Template |

## Dependencies
Run **12a_downloadSelectedCutoutsFits.ipynb** first.

---
- **Author:** Sylvie Dagoret-Campagne — IJCLab/IN2P3/CNRS — Université Paris-Saclay
- **Created:** 2026-06-11  |  **Updated:** 2026-06-14
- **Subject:** Fink/LSST DIA dipole hypothesis — FITS cutout viewer + WCS orientation

## 1. Imports

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize, TwoSlopeNorm
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u
import astropy

warnings.filterwarnings("ignore")
print(f"numpy   {np.__version__}")
print(f"pandas  {pd.__version__}")
print(f"astropy {astropy.__version__}")

In [ ]:
%matplotlib inline

## 2. User parameters

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# USER PARAMETERS  ← edit here
# ─────────────────────────────────────────────────────────────────────────────
objsid = {
    0: 313985344866353157,  # rank 2  COSMOS
    1: 313853517840777344,  # rank 3  COSMOS
    2: 313972182542712999,  # rank 4  COSMOS
    3: 313871013109563545,  # rank 5  COSMOS
    4: 313871013420466334,  # rank 6  COSMOS
    5: 313998569477505082,  # rank 5  COSMOS
    6: 313994141002367046,  # rank 6  COSMOS  QSO
    7: 313888627167330394,  # rank 1  COSMOS
}
DIAOBJECT_IDX = 5  # ← change to switch object
DIAOBJECT_ID = objsid[DIAOBJECT_IDX]

MJD_MIN = None  # float or None
MJD_MAX = None
BANDS_FILTER = None  # e.g. ['r','i'] or None
MAX_ROWS_PER_BAND = None  # int or None
CMAP_SCI = "gray"
CMAP_DIFF = "RdBu_r"
SAVE_FIGS = True
DIR_FIGS = "figs_DIPOLES_12b_fits"

# ── Derived paths ─────────────────────────────────────────────────────────────
DIR_CUTOUTS = f"fullcutouts_fits_{DIAOBJECT_ID}"
FILE_MANIFEST = os.path.join(DIR_CUTOUTS, "manifest_src.csv")
FILE_MANIFEST_FP = os.path.join(DIR_CUTOUTS, "manifest_fp.csv")
FILE_DIPOLE_STATS = os.path.join("data_DIPOLES_03b", "topranked_objects_dipoles.csv")
FILE_DIPOLE_ANGLES = os.path.join("data_DIPOLES_03b", "dipole_angle_stability.csv")

AB_FLUX_ZERO = 3631e9
BAND_ORDER = list("ugrizy")
BAND_COLORS = {"u": "#9b59b6", "g": "#2ecc71", "r": "#e74c3c", "i": "#e67e22", "z": "#3498db", "y": "#795548"}

os.makedirs(DIR_FIGS, exist_ok=True)
print(f"diaObjectId : {DIAOBJECT_ID}  (idx {DIAOBJECT_IDX})")
print(f"FITS dir    : {os.path.abspath(DIR_CUTOUTS)}")
print(f"Figures     : {os.path.abspath(DIR_FIGS)}")

## 3. Utility functions

In [ ]:
def flux_to_mag_AB(f_nJy):
    try:
        f = float(f_nJy)
        return -2.5 * np.log10(f / AB_FLUX_ZERO) if (np.isfinite(f) and f > 0) else np.nan
    except:
        return np.nan


def symvlim(arr, pct=99.5):
    fin = arr[np.isfinite(arr)]
    return max(float(np.percentile(np.abs(fin), pct)) if len(fin) else 1.0, 1e-9)


def load_fits(src_id, band, kind):
    """Return (data float32, header) or (None, None)."""
    p = os.path.join(DIR_CUTOUTS, "cutouts", f"{src_id}_{band}_{kind}.fits")
    if not os.path.exists(p):
        return None, None
    with fits.open(p) as h:
        return h[0].data.squeeze().astype(np.float32), h[0].header.copy()


def parse_bool(v):
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)) and np.isfinite(float(v)):
        return bool(int(v))
    if isinstance(v, str):
        return v.strip().lower() in ("true", "1", "yes")
    return False


def mjd_to_date(mjd):
    try:
        return Time(float(mjd), format="mjd", scale="tai").isot[:10]
    except:
        return "?"


def fmt_flag(v, t="✓", f="✗"):
    return t if parse_bool(v) else f


def fmt_float(v, fmt=".3f", na="—"):
    try:
        x = float(v)
        return f"{x:{fmt}}" if np.isfinite(x) else na
    except:
        return na


def getminmaxvals(s):
    y = s.dropna()
    if len(y) >= 5:
        q1, q3 = y.quantile([0.25, 0.75])
        iqr = q3 - q1
        yi = y[(y >= q1 - 1.5 * iqr) & (y <= q3 + 1.5 * iqr)] if iqr > 0 else y
        if len(yi):
            lo, hi = yi.min(), yi.max()
            pad = 0.5 * (hi - lo) if hi > lo else max(1.0, 0.1 * abs(hi))
            return lo - pad, hi + pad
    return None, None


print("Utility functions OK")

## 4. WCS orientation helper functions

### North & East from the CD matrix

$$\mathbf{CD}\,\begin{pmatrix}\Delta x\\\Delta y\end{pmatrix}=\begin{pmatrix}\Delta\alpha\cos\delta\\\Delta\delta\end{pmatrix}$$

Pixel direction of a sky unit-vector $(u_E,u_N)$: $\mathbf{CD}^{-1}(u_E,u_N)^T$, then normalised.

### Zenith direction — parallactic angle η

$$\tan\eta=\frac{\sin H}{\cos\delta\,\tan\phi-\sin\delta\cos H}$$

$H$ = hour angle, $\phi=-30.24°$ (Rubin).  Zenith sky unit-vector: $(\sin\eta,\cos\eta)$.

### Dipole direction

`DIPPA = r:dipoleAngle` [deg, CCW from East pixel axis].  
Notebook 05b confirms `r:dipoleAngle ≈ η`, so it **directly** gives the zenith direction — **no 90° offset**.  
Sky vector: $(\cos\text{DIPPA},\sin\text{DIPPA})$.


In [ ]:
# ── Rubin site ─────────────────────────────────────────────────────────────
RUBIN_LAT_DEG = -30.244728
RUBIN_LON_DEG = -70.749417
RUBIN_HEIGHT_M = 2647.0
RUBIN_LOC = EarthLocation(lat=RUBIN_LAT_DEG * u.deg, lon=RUBIN_LON_DEG * u.deg, height=RUBIN_HEIGHT_M * u.m)


# ── Parallactic angle (Meeus formula) ──────────────────────────────────────
def parallactic_angle_rad(obs_time, ra_deg, dec_deg, location):
    """Return parallactic angle q [rad].  Positive when zenith is E of N."""
    target = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
    lst = obs_time.sidereal_time("apparent", longitude=location.lon)
    ha = (lst - target.ra).to(u.rad)
    lat = location.lat.to(u.rad)
    dec = target.dec.to(u.rad)
    q = np.arctan2(np.sin(ha), np.tan(lat) * np.cos(dec) - np.sin(dec) * np.cos(ha))
    return float(q.to_value(u.rad))  # strip Quantity unit → plain float


# ── Direction vectors in pixel space via PC matrix (robust) ────────────────
def direction_vectors_wcs(wcs_obj, ra_deg, dec_deg, mjd_tai):
    """
    Return North, East, Zenith pixel unit-vectors and observing geometry.

    Uses the CD/PC matrix directly from the WCS object — robust against
    missing or empty CTYPE keywords (which occur when cutouts are downloaded
    without FLAG_SET_CTYPE).  The PC matrix + CRVAL/CRPIX are always present.

    CD = diag(CDELT) @ PC   [deg/pix]
    North pixel direction : CD^{-1} @ [0, +1]  (= +Dec in sky)
    East  pixel direction : CD^{-1} @ [+1, 0]  (= +RA*cosDec in sky)

    Returns
    -------
    north_pix, east_pix, zenith_pix : ndarray (2,)  unit vectors in pixel space
    q_deg  : float  parallactic angle [deg]
    alt_deg: float  altitude [deg]
    """
    # Build CD matrix from WCS internals (always available)
    # wcs_obj.wcs.cd  exists if CD keywords; otherwise build from PC + CDELT
    try:
        CD = wcs_obj.wcs.cd.copy()  # shape (2,2), deg/pix
    except AttributeError:
        pc = wcs_obj.wcs.get_pc()  # shape (2,2), dimensionless
        cdelt = wcs_obj.wcs.cdelt  # shape (2,), deg/pix
        CD = np.diag(cdelt) @ pc

    CDinv = np.linalg.inv(CD)

    # North: +Dec direction in pixel space
    north_pix = CDinv @ np.array([0.0, 1.0])
    north_pix /= np.linalg.norm(north_pix)

    # East: +RA*cosDec direction in pixel space
    east_pix = CDinv @ np.array([1.0, 0.0])
    east_pix /= np.linalg.norm(east_pix)

    # Parallactic angle → zenith direction
    t = Time(mjd_tai, format="mjd", scale="tai")
    src = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
    aa = src.transform_to(AltAz(obstime=t, location=RUBIN_LOC))
    alt_deg = float(aa.alt.deg)
    q_rad = float(parallactic_angle_rad(t, ra_deg, dec_deg, RUBIN_LOC))
    q_deg = float(np.rad2deg(q_rad))

    # Zenith = North rotated by q toward East
    zenith_pix = np.cos(q_rad) * north_pix + np.sin(q_rad) * east_pix
    zenith_pix /= np.linalg.norm(zenith_pix)

    return north_pix, east_pix, zenith_pix, q_deg, alt_deg


def dipole_pixel_vector_from_pa(dippa_deg, north_pix, east_pix):
    """
    Dipole axis unit-vector in pixel space.
    dippa_deg = r:dipoleAngle in the North-toward-East PA convention
    (PA=0 → North, PA=90 → East).
    """
    pa = np.deg2rad(dippa_deg % 360.0)
    dp = np.sin(pa) * east_pix + np.cos(pa) * north_pix
    return dp / np.linalg.norm(dp)


def draw_orientation_arrows(
    ax,
    x0,
    y0,
    shape,
    north_pix,
    east_pix,
    zenith_pix=None,
    q_deg=None,
    dip_pix=None,
    dip_len_arcsec=None,
    arrow_frac=0.30,
):
    """
    Overlay N/E/Z direction arrows and an optional dipole segment on a
    WCSAxes subplot, using ax.arrow() + ax.text() with
    transform=ax.get_transform("pixel").

    This matches exactly the approach used in 12_fetch_onecutouts.ipynb and
    avoids the ax.annotate(xycoords=transform) bug on WCSAxes.

    Parameters
    ----------
    ax         : WCSAxes subplot
    x0, y0     : float  arrow anchor in pixel coordinates (image centre)
    shape      : (ny, nx)
    north_pix,
    east_pix   : ndarray (2,)  pixel unit-vectors for N and E
    zenith_pix : ndarray (2,) or None
    q_deg      : float or None  parallactic angle [deg], for label
    dip_pix    : ndarray (2,) or None  dipole axis unit-vector
    dip_len_arcsec : float or None  r:dipoleLength [arcsec], for label
    arrow_frac : float  arrow length as fraction of min(ny, nx)
    """
    ny, nx = shape
    scale = arrow_frac * min(ny, nx)
    pt = ax.get_transform("pixel")

    # arrow kwargs shared by all directions
    aw = max(0.3, scale * 0.04)  # shaft width
    hw = aw * 3  # head width
    hl = aw * 4  # head length
    kw_arr = dict(
        width=aw,
        head_width=hw,
        head_length=hl,
        length_includes_head=True,
        transform=pt,
    )
    loff = 1.22  # label offset factor beyond arrow tip
    kw_txt = dict(fontsize=8, fontweight="bold", ha="center", va="center", transform=pt)

    # ── North — red ────────────────────────────────────────────────────────
    dx_n, dy_n = north_pix * scale
    ax.arrow(x0, y0, dx_n, dy_n, color="red", **kw_arr)
    ax.text(x0 + dx_n * loff, y0 + dy_n * loff, "N", color="red", **kw_txt)

    # ── East — blue ────────────────────────────────────────────────────────
    dx_e, dy_e = east_pix * scale
    ax.arrow(x0, y0, dx_e, dy_e, color="blue", **kw_arr)
    ax.text(x0 + dx_e * loff, y0 + dy_e * loff, "E", color="blue", **kw_txt)

    # ── Zenith — yellow ────────────────────────────────────────────────────
    if zenith_pix is not None:
        dx_z, dy_z = zenith_pix * scale
        qlbl = f"\nη={q_deg:+.0f}°" if q_deg is not None else ""
        ax.arrow(x0, y0, dx_z, dy_z, color="yellow", **kw_arr)
        ax.text(
            x0 + dx_z * loff,
            y0 + dy_z * loff,
            f"Z{qlbl}",
            color="yellow",
            fontsize=7,
            fontweight="bold",
            ha="center",
            va="center",
            transform=pt,
        )

    # ── Dipole axis — cyan bidirectional ──────────────────────────────────
    if dip_pix is not None:
        dscale = scale * 0.85
        dx_d, dy_d = dip_pix * dscale
        lbl = f'\n{dip_len_arcsec:.2f}"' if dip_len_arcsec is not None else ""
        # draw two opposite arrows to mimic <->
        ax.arrow(x0, y0, dx_d, dy_d, color="cyan", **kw_arr)
        ax.arrow(x0, y0, -dx_d, -dy_d, color="cyan", **kw_arr)
        ax.text(
            x0 + dx_d * loff,
            y0 + dy_d * loff,
            f"Dip{lbl}",
            color="cyan",
            fontsize=7,
            fontweight="bold",
            ha="center",
            va="center",
            transform=pt,
        )


print("WCS orientation helpers OK  (ax.arrow-based, anchor=image-centre)")
print(f"Rubin: lat={RUBIN_LAT_DEG} deg  lon={RUBIN_LON_DEG} deg  h={RUBIN_HEIGHT_M} m")

## 5. Load dipole statistics from notebook 03b

In [ ]:
obj_field = obj_rank = obj_label = obj_label_clf = obj_gaia_name = obj_simbad = "?"
obj_ra = obj_dec = obj_dipole_frac = np.nan
obj_nDiaSources = obj_n_src = obj_n_dipoles = 0
obj_n_dip_per_band = {}

if os.path.exists(FILE_DIPOLE_STATS):
    dfs = pd.read_csv(FILE_DIPOLE_STATS)
    hit = dfs[dfs["diaObjectId"].astype(str) == str(DIAOBJECT_ID)]
    if len(hit):
        r = hit.iloc[0]
        obj_field = str(r.get("field", "?"))
        obj_ra = float(r.get("ra", np.nan))
        obj_dec = float(r.get("dec", np.nan))
        obj_nDiaSources = int(r.get("nDiaSources", 0))
        obj_n_src = int(r.get("n_src", 0))
        obj_n_dipoles = int(r.get("n_dipoles", 0))
        obj_dipole_frac = float(r.get("dipole_fraction", np.nan))
        obj_rank = str(r.get("rank", "?"))
        obj_label = str(r.get("label", "?"))
        obj_label_clf = str(r.get("label_clf", "?"))
        obj_gaia_name = str(r.get("gaia_name", "?"))
        obj_simbad = str(r.get("simbad", "?"))
        for b in BAND_ORDER:
            c = f"n_dip_{b}"
            obj_n_dip_per_band[b] = int(r.get(c, 0)) if c in r.index else 0
    else:
        print(f"  WARNING: {DIAOBJECT_ID} not in {FILE_DIPOLE_STATS}")
else:
    print(f"  WARNING: {FILE_DIPOLE_STATS} not found (run notebook 03b)")

print(
    f"Object {DIAOBJECT_ID}: field={obj_field}  "
    f"n_src={obj_n_src}  n_dip={obj_n_dipoles}  frac={obj_dipole_frac:.3f}"
)
print(f"Gaia={obj_gaia_name}  Simbad={obj_simbad}")

## 6. Load dipole angle stability from notebook 03b

In [ ]:
obj_angle_mean_deg = obj_angle_circ_std_deg = obj_length_median_arcsec = np.nan
obj_angle_cstd_per_band = {}

if os.path.exists(FILE_DIPOLE_ANGLES):
    dfa = pd.read_csv(FILE_DIPOLE_ANGLES)
    hit = dfa[dfa["diaObjectId"].astype(str) == str(DIAOBJECT_ID)]
    if len(hit):
        r = hit.iloc[0]
        obj_angle_mean_deg = float(r.get("angle_mean_deg", np.nan))
        obj_angle_circ_std_deg = float(r.get("angle_circ_std_deg", np.nan))
        obj_length_median_arcsec = float(r.get("length_median_arcsec", np.nan))
        for b in BAND_ORDER:
            c = f"angle_cstd_{b}"
            try:
                obj_angle_cstd_per_band[b] = float(r.get(c, np.nan))
            except:
                obj_angle_cstd_per_band[b] = np.nan
    else:
        print(f"  WARNING: {DIAOBJECT_ID} not in {FILE_DIPOLE_ANGLES}")
else:
    print(f"  WARNING: {FILE_DIPOLE_ANGLES} not found (run notebook 03b)")

print(
    f"angle_mean={fmt_float(obj_angle_mean_deg)} deg  "
    f"circ_std={fmt_float(obj_angle_circ_std_deg)} deg  "
    f"len_median={fmt_float(obj_length_median_arcsec, '.4f')} arcsec"
)

## 7. Load manifest

In [ ]:
if not os.path.exists(FILE_MANIFEST):
    raise FileNotFoundError(f"{FILE_MANIFEST} not found.\nRun 12a_downloadSelectedCutoutsFits.ipynb first.")

df = pd.read_csv(FILE_MANIFEST)

for bc in ["r:isDipole", "r:isNegative", "r:dipoleFitAttempted"]:
    if bc in df.columns:
        df[bc] = df[bc].fillna(False).apply(parse_bool)

df["isDipole"] = df["r:isDipole"] if "r:isDipole" in df.columns else False
df["isNegative"] = df["r:isNegative"] if "r:isNegative" in df.columns else False
df["dipoleFitAttempted"] = df["r:dipoleFitAttempted"] if "r:dipoleFitAttempted" in df.columns else False

df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)

mask = pd.Series(True, index=df.index)
if MJD_MIN is not None:
    mask &= df["r:midpointMjdTai"] >= MJD_MIN
if MJD_MAX is not None:
    mask &= df["r:midpointMjdTai"] <= MJD_MAX
df = df[mask].reset_index(drop=True)

if BANDS_FILTER is not None:
    df = df[df["r:band"].isin(BANDS_FILTER)].reset_index(drop=True)

df["r:band"] = pd.Categorical(df["r:band"], categories=BAND_ORDER, ordered=True)
df = df.sort_values(["r:band", "r:visit"]).reset_index(drop=True)

nd = df["isDipole"].sum()
print(f"Manifest: {len(df)} diaSources  |  {nd} isDipole ({100 * nd / max(len(df), 1):.1f}%)")
print(f"Bands: {sorted(df['r:band'].unique())}")
df[["r:diaSourceId", "r:band", "r:visit", "r:psfFlux", "isDipole", "r:dipoleLength", "r:dipoleAngle"]].head(8)

## 8. Load forced photometry

In [ ]:
df_fp = pd.DataFrame()
if os.path.exists(FILE_MANIFEST_FP):
    df_fp = pd.read_csv(FILE_MANIFEST_FP).sort_values("r:midpointMjdTai").reset_index(drop=True)
    print(f"FP: {len(df_fp)} pts  bands={sorted(df_fp['r:band'].unique())}")
else:
    print(f"WARNING: {FILE_MANIFEST_FP} not found — LC shows diaSources only")

all_mjd = df["r:midpointMjdTai"].values
t0 = float(np.nanmin(all_mjd)) if len(all_mjd) else 0.0
print(f"t0 = MJD {t0:.4f}  ({mjd_to_date(t0)})")

## 9. Per-panel helpers (info text, light curve)

In [ ]:
def build_info_text(row, q_deg=None, alt_deg=None):
    """Monospace info block for the top-left panel."""
    src_id = int(row["r:diaSourceId"])
    mjd = row["r:midpointMjdTai"]
    mag = flux_to_mag_AB(row.get("r:psfFlux"))
    lines = [
        f"diaSourceId : {src_id}",
        f"diaObjectId : {DIAOBJECT_ID}",
        f"band={row['r:band']}  visit={row.get('r:visit', '?')}  det={row.get('r:detector', '?')}",
        f"MJD={mjd:.4f}  ({mjd_to_date(mjd)})",
        f"RA={fmt_float(row.get('r:ra'), '.5f')}  Dec={fmt_float(row.get('r:dec'), '.5f')}",
        "",
        f"psfFlux  = {fmt_float(row.get('r:psfFlux'), '.1f')} nJy  (mag={fmt_float(mag, '.3f')})",
        f"sciFlux  = {fmt_float(row.get('r:scienceFlux'), '.1f')} nJy",
        f"tplFlux  = {fmt_float(row.get('r:templateFlux'), '.1f')} nJy",
        f"SNR      = {fmt_float(row.get('r:snr'), '.1f')}  rel={fmt_float(row.get('r:reliability'), '.3f')}",
        "",
        f"isDipole={fmt_flag(row.get('r:isDipole', False))}  "
        f"isNeg={fmt_flag(row.get('r:isNegative', False))}  "
        f"fitAtt={fmt_flag(row.get('r:dipoleFitAttempted', False))}",
        f"dipoleFluxDiff = {fmt_float(row.get('r:dipoleFluxDiff'), '.1f')} nJy",
        f"dipoleMeanFlux = {fmt_float(row.get('r:dipoleMeanFlux'), '.1f')} nJy",
        f"dipoleLength   = {fmt_float(row.get('r:dipoleLength'), '.3f')} arcsec",
        f"dipoleAngle    = {fmt_float(row.get('r:dipoleAngle'), '.1f')} deg  (= DIPPA)",
        f"dipoleChi2     = {fmt_float(row.get('r:dipoleChi2'), '.2f')}",
        "",
        f"field={obj_field}  rank={obj_rank}",
        f"dipole_frac={fmt_float(obj_dipole_frac, '.3f')}",
        f"Gaia  : {obj_gaia_name}",
        f"Simbad: {obj_simbad}",
    ]
    if q_deg is not None:
        lines += ["", f"eta (parallactic) = {q_deg:+.1f} deg"]
    if alt_deg is not None:
        lines += [f"Alt = {alt_deg:.2f} deg"]
    return "\n".join(lines)


def plot_lightcurve(ax, band, current_mjd):
    """psfFlux LC with FP overlay; current visit marked with dashed line."""
    sub = df[df["r:band"] == band].sort_values("r:midpointMjdTai")
    col = BAND_COLORS.get(band, "gray")
    ax.errorbar(
        sub["r:midpointMjdTai"] - t0,
        sub["r:psfFlux"],
        yerr=sub.get("r:psfFluxErr", None),
        fmt="o",
        color=col,
        ms=3,
        lw=0.8,
        alpha=0.7,
        label="diaSource",
    )
    ymin, ymax = getminmaxvals(sub["r:psfFlux"])
    if not df_fp.empty:
        fp = df_fp[df_fp["r:band"] == band].sort_values("r:midpointMjdTai")
        ax.errorbar(
            fp["r:midpointMjdTai"] - t0,
            fp["r:psfFlux"],
            yerr=fp.get("r:psfFluxErr", None),
            fmt="^",
            color=col,
            ms=4,
            lw=0.6,
            alpha=0.35,
            label="fp",
        )
    ax.axvline(current_mjd - t0, color="k", lw=1.0, ls="--", alpha=0.8)
    ax.set_xlabel(f"MJD - {t0:.0f}  [days]", fontsize=10)
    ax.set_ylabel("psfFlux [nJy]", fontsize=10)
    ax.set_title(f"band={band}", fontsize=10)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=8, loc="upper left")
    if ymin is not None:
        ax.set_ylim(ymin, ymax)


print("Panel helpers OK")

## 10. Main visualisation loop — 2 × 3 grid per diaSource

| | col 0 | col 1 | col 2 |
|---|---|---|---|
| **row 0** | Info panel | Science + N/E/Z | Template + N/E/Z |
| **row 1** | Light curve | DIA Diff + N/E/Z/Dip | Sci−Tpl + N/E/Z/Dip |

Colour code: **cyan** N/E · **red** Zenith η · **orange** ↔ Dipole axis


In [ ]:
if MAX_ROWS_PER_BAND is not None:
    df_plot = (
        df.groupby("r:band", group_keys=False)
        .apply(lambda g: g.head(MAX_ROWS_PER_BAND))
        .reset_index(drop=True)
    )
else:
    df_plot = df

zscaler = ZScaleInterval()
print(f"Plotting {len(df_plot)} diaSources ...")

for idx, row in df_plot.iterrows():
    src_id = int(row["r:diaSourceId"])
    band = row["r:band"]
    mjd = row["r:midpointMjdTai"]
    is_dip = parse_bool(row.get("r:isDipole", False))
    ra_src = row.get("r:ra", np.nan)
    dec_src = row.get("r:dec", np.nan)
    dip_len = row.get("r:dipoleLength", np.nan)
    dip_ang = row.get("r:dipoleAngle", np.nan)  # = DIPPA

    # ── Load FITS ────────────────────────────────────────────────────────────
    arr_sci, hdr_sci = load_fits(src_id, band, "Science")
    arr_tpl, hdr_tpl = load_fits(src_id, band, "Template")
    arr_diff, hdr_diff = load_fits(src_id, band, "Difference")

    if arr_sci is None and arr_tpl is None and arr_diff is None:
        print(f"  [{idx + 1}] {src_id} {band} — no FITS, skip")
        continue

    hdr_wcs = hdr_sci if hdr_sci is not None else hdr_diff

    # ── WCS & orientation ────────────────────────────────────────────────────
    # Force CTYPE to TAN so WCS projection is always valid (Fink cutouts
    # may have empty CTYPE when downloaded without FLAG_SET_CTYPE)
    if hdr_wcs is not None:
        hdr_wcs_tan = hdr_wcs.copy()
        hdr_wcs_tan["CTYPE1"] = "RA---TAN"
        hdr_wcs_tan["CTYPE2"] = "DEC--TAN"
        wcs_obj = WCS(hdr_wcs_tan)
    else:
        wcs_obj = None

    north_pix = east_pix = zenith_pix = dip_pix = None
    q_deg = alt_deg = None

    if wcs_obj is not None and np.isfinite(ra_src) and np.isfinite(dec_src) and np.isfinite(mjd):
        try:
            north_pix, east_pix, zenith_pix, q_deg, alt_deg = direction_vectors_wcs(
                wcs_obj, float(ra_src), float(dec_src), float(mjd)
            )
            print(f"  η={q_deg:+.2f}° Alt={alt_deg:.1f}°")
        except Exception as e:
            print(f"  direction_vectors_wcs warning: {e}")
    _ang = (
        float(dip_ang)
        if dip_ang is not None and not (isinstance(dip_ang, float) and np.isnan(dip_ang))
        else np.nan
    )
    if is_dip and np.isfinite(_ang) and north_pix is not None:
        dip_pix = dipole_pixel_vector_from_pa(_ang, north_pix, east_pix)

    # ── Source pixel coords ──────────────────────────────────────────────────
    src_xp = src_yp = None
    if wcs_obj is not None and np.isfinite(ra_src) and np.isfinite(dec_src):
        try:
            src_xp, src_yp = wcs_obj.all_world2pix([[ra_src, dec_src]], 0)[0]
        except:
            pass

    # ── Colour scales ────────────────────────────────────────────────────────
    if arr_sci is not None and arr_tpl is not None:
        cst = np.concatenate([arr_sci.ravel(), arr_tpl.ravel()])
    elif arr_sci is not None:
        cst = arr_sci.ravel()
    elif arr_tpl is not None:
        cst = arr_tpl.ravel()
    else:
        cst = np.array([0.0, 1.0])
    vlo, vhi = zscaler.get_limits(cst[np.isfinite(cst)])
    norm_st = Normalize(vmin=vlo, vmax=vhi)

    arr_smt = (arr_sci - arr_tpl) if (arr_sci is not None and arr_tpl is not None) else None
    cdiff = ([arr_diff.ravel()] if arr_diff is not None else []) + (
        [arr_smt.ravel()] if arr_smt is not None else []
    )
    vm_diff = symvlim(np.concatenate(cdiff)) if cdiff else 1.0
    vm_diff = 1.3 * vm_diff
    norm_diff = TwoSlopeNorm(vmin=-vm_diff, vcenter=0.0, vmax=vm_diff)

    # ── Figure 2x3 ───────────────────────────────────────────────────────────
    pkw = {"projection": wcs_obj} if wcs_obj is not None else {}
    fig = plt.figure(figsize=(15, 6.5))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.28)
    ax_info = fig.add_subplot(gs[0, 0])
    ax_sci = fig.add_subplot(gs[0, 1], **pkw)
    ax_tpl = fig.add_subplot(gs[0, 2], **pkw)
    ax_lc = fig.add_subplot(gs[1, 0])
    ax_diff = fig.add_subplot(gs[1, 1], **pkw)
    ax_smt = fig.add_subplot(gs[1, 2], **pkw)

    dc = "#c0392b" if is_dip else "#2c3e50"  # dipole colour for titles / text

    def wcs_cosm(ax, title, fs=9):
        ax.set_title(title, fontsize=fs, color=dc)
        if wcs_obj is not None:
            ax.set_xlabel("Right Ascension", fontsize=9)
            ax.set_ylabel("Declination", fontsize=9)
            ax.coords["ra"].set_major_formatter("dd:mm:ss")
            ax.coords["dec"].set_major_formatter("dd:mm:ss")
            ax.coords["ra"].set_ticklabel(size=7)
            ax.coords["dec"].set_ticklabel(size=7)
            # ax.coords.grid(True, color="white", ls="dotted", lw=0.6, alpha=0.5)
            ax.coords.grid(True, color="lightgreen", ls="-.", lw=1.5, alpha=1.0)
        else:
            ax.axis("off")

    def add_overlays(ax, arr, with_dipole=False):
        ny, nx = arr.shape
        # Anchor arrows at the image centre
        cx, cy = nx / 2.0, ny / 2.0
        if src_xp is not None:
            ax.plot(
                src_xp,
                src_yp,
                "r+",
                ms=14,
                mew=2,
                transform=ax.get_transform("pixel") if wcs_obj else ax.transData,
                zorder=5,
            )
        if north_pix is not None:
            dl = (
                float(dip_len)
                if (with_dipole and dip_pix is not None and np.isfinite(float(dip_len)))
                else None
            )
            draw_orientation_arrows(
                ax,
                cx,
                cy,
                (ny, nx),
                north_pix=north_pix,
                east_pix=east_pix,
                zenith_pix=zenith_pix,
                q_deg=q_deg,
                dip_pix=dip_pix if with_dipole else None,
                dip_len_arcsec=dl,
                arrow_frac=0.30,
            )

    # (0,0) Info panel
    ax_info.axis("off")
    ax_info.text(
        0.04,
        0.97,
        build_info_text(row, q_deg=q_deg, alt_deg=alt_deg),
        transform=ax_info.transAxes,
        va="top",
        ha="left",
        fontsize=7,
        fontfamily="monospace",
        color=dc,
    )

    # (0,1) Science
    if arr_sci is not None:
        im = ax_sci.imshow(arr_sci, origin="lower", cmap=CMAP_SCI, norm=norm_st)
        add_overlays(ax_sci, arr_sci, with_dipole=False)
        fig.colorbar(im, ax=ax_sci, fraction=0.046, pad=0.04)
    else:
        ax_sci.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax_sci.transAxes)
    wcs_cosm(ax_sci, f"Science  {band}  MJD={mjd:.3f}")

    # (0,2) Template
    if arr_tpl is not None:
        im = ax_tpl.imshow(arr_tpl, origin="lower", cmap=CMAP_SCI, norm=norm_st)
        add_overlays(ax_tpl, arr_tpl, with_dipole=False)
        fig.colorbar(im, ax=ax_tpl, fraction=0.046, pad=0.04)
    else:
        ax_tpl.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax_tpl.transAxes)
    wcs_cosm(ax_tpl, "Template")

    # (1,0) Light curve
    try:
        plot_lightcurve(ax_lc, band, mjd)
    except Exception as exc:
        ax_lc.text(
            0.5, 0.5, f"LC error:\n{exc}", ha="center", va="center", transform=ax_lc.transAxes, fontsize=7
        )

    # (1,1) DIA Difference
    if arr_diff is not None:
        im = ax_diff.imshow(arr_diff, origin="lower", cmap=CMAP_DIFF, norm=norm_diff)
        add_overlays(ax_diff, arr_diff, with_dipole=True)
        wcs_cosm(ax_diff, f"DIA Difference{'  [DIPOLE]' if is_dip else ''}")
        fig.colorbar(im, ax=ax_diff, fraction=0.046, pad=0.04)
    else:
        ax_diff.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax_diff.transAxes)
        wcs_cosm(ax_diff, "DIA Difference")

    # (1,2) Sci - Template (computed)
    if arr_smt is not None:
        im = ax_smt.imshow(arr_smt, origin="lower", cmap=CMAP_DIFF, norm=norm_diff)
        add_overlays(ax_smt, arr_smt, with_dipole=False)  # dipole shown on Diff only
        wcs_cosm(ax_smt, "Sci - Template  (computed)")
        fig.colorbar(im, ax=ax_smt, fraction=0.046, pad=0.04)
    else:
        ax_smt.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax_smt.transAxes)
        wcs_cosm(ax_smt, "Sci - Template  (computed)")

    # Suptitle
    dstr = f'isDipole={is_dip}  L={fmt_float(dip_len, ".3f")}"  DIPPA={fmt_float(dip_ang, ".1f")} deg'
    qstr = f"{q_deg:.1f}" if q_deg is not None else "n/a"
    fig.suptitle(
        f"obj={DIAOBJECT_ID}  src={src_id}  [{idx + 1}/{len(df_plot)}]  |  {dstr}\n"
        f"Red: N · Blue: E · Yellow: Zenith (η)   Cyan: Dipole axis (diff only)   η={qstr}°",
        fontsize=11,
        color=BAND_COLORS.get(band, "black"),
    )

    if SAVE_FIGS:
        fn = f"cutout_fits_{DIAOBJECT_ID}_{src_id}_{band}"
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(DIR_FIGS, f"{fn}.{ext}"), bbox_inches="tight", dpi=130)
    plt.show()
    plt.close(fig)

print("Done.")